In [1]:
import os, shutil

BASE_DIR = "/kaggle/working/spectral_v2"
DIRS = {
    "generations": f"{BASE_DIR}/generations", "extractions": f"{BASE_DIR}/extractions",
    "features": f"{BASE_DIR}/features", "results": f"{BASE_DIR}/results", "logs": f"{BASE_DIR}/logs",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print(os.listdir("/kaggle/input/notebooks/aishidev"))

['final-spectral-research-project-arc-analysis']


In [2]:
import shutil

SOURCE = "/kaggle/input/notebooks/aishidev/final-spectral-research-project-arc-analysis/spectral_v2"
shutil.copytree(SOURCE, BASE_DIR, dirs_exist_ok=True)

print("ARC files:", len(os.listdir(f"{DIRS['generations']}/arc_challenge")))

ARC files: 300


In [3]:
# ============================================================
# ARC-CHALLENGE — Feature computation at scale
# Mirrors GSM8K's Checkpoint 8: reads saved tensors, builds
# graphs, computes spectral metrics, produces the consolidated
# per-example x per-layer feature table.
# No GPU needed -- pure numpy computation.
# ============================================================

import os, json
import numpy as np
import pandas as pd

ARC_DIR = f"{DIRS['generations']}/arc_challenge"

# --- Graph-building function (same as GSM8K's) ---
def build_layer_graphs_np(attentions, eps=1e-8):
    n_layers = attentions.shape[0]
    layer_graphs = []
    for l in range(n_layers):
        A = attentions[l].astype(np.float64)
        heads, seq, _ = A.shape
        W_per_head = 0.5 * (A + A.transpose(0, 2, 1))
        head_mass = W_per_head.sum(axis=(1, 2))
        head_weights = head_mass / (head_mass.sum() + eps)
        W = np.tensordot(head_weights, W_per_head, axes=(0, 0))
        D = np.diag(W.sum(axis=1))
        L = D - W
        layer_graphs.append({"layer": l, "L": L})
    return layer_graphs

# --- Spectral metrics function (same as GSM8K's) ---
def compute_spectral_metrics_np(L, hidden_state, eps=1e-10):
    n = L.shape[0]
    eigvals, eigvecs = np.linalg.eigh(L)
    eigvals = np.clip(eigvals, 0, None)

    fiedler = eigvals[1] if n > 1 else 0.0

    eigval_sum = eigvals.sum()
    if eigval_sum > eps:
        p = eigvals / eigval_sum
        p_nonzero = p[p > eps]
        spectral_entropy = -np.sum(p_nonzero * np.log(p_nonzero))
        spectral_entropy = spectral_entropy / np.log(n) if n > 1 else 0.0
    else:
        spectral_entropy = 0.0

    x = hidden_state.astype(np.float64)
    signal = np.linalg.norm(x, axis=1)
    signal = (signal - signal.mean()) / (signal.std() + eps)

    dirichlet_energy = float(signal @ L @ signal)

    signal_proj = eigvecs.T @ signal
    energy = signal_proj ** 2
    total_energy = energy.sum()
    cutoff_idx = n // 2
    hfer = float(energy[cutoff_idx:].sum() / (total_energy + eps))

    smoothness = float(1.0 / (1.0 + dirichlet_energy))

    return {
        "fiedler": float(fiedler), "spectral_entropy": float(spectral_entropy),
        "hfer": hfer, "smoothness": smoothness, "dirichlet_energy": dirichlet_energy,
    }

# --- Run across all 150 ARC examples ---
all_rows = []
failed_examples = []

json_files = [f for f in os.listdir(ARC_DIR) if f.endswith(".json")]
print(f"Processing {len(json_files)} ARC examples...")

for i, fname in enumerate(json_files):
    example_id = fname.replace(".json", "")
    json_path = f"{ARC_DIR}/{fname}"
    npz_path = f"{ARC_DIR}/{example_id}_tensors.npz"

    try:
        with open(json_path) as f:
            record = json.load(f)

        tensors = np.load(npz_path)
        attentions = tensors["attentions"]
        hidden_states = tensors["hidden_states"]

        layer_graphs = build_layer_graphs_np(attentions)

        for lg in layer_graphs:
            layer_idx = lg["layer"]
            hs = hidden_states[layer_idx]
            metrics = compute_spectral_metrics_np(lg["L"], hs)

            row = {
                "example_id": example_id,
                "layer": layer_idx,
                "is_correct": record["is_correct"],
                "response_len": record["response_len"],
                "prompt_len": record["prompt_len"],
                "total_len": record["total_len"],
                **metrics,
            }
            all_rows.append(row)

        if (i + 1) % 25 == 0:
            print(f"  [{i+1}/{len(json_files)}] processed")

    except Exception as e:
        failed_examples.append((example_id, str(e)))
        print(f"FAILED on {example_id}: {e}")
        continue

print(f"\nDone. {len(json_files) - len(failed_examples)} examples processed successfully, "
      f"{len(failed_examples)} failed.")

# --- Save the consolidated feature table ---
features_df = pd.DataFrame(all_rows)
print(f"\nFeature table shape: {features_df.shape}")
expected_rows = (len(json_files) - len(failed_examples)) * 24
print(f"Expected: {len(json_files) - len(failed_examples)} examples x 24 layers = {expected_rows} rows")

assert features_df.shape[0] == expected_rows, "STOP: row count doesn't match expected examples x 24 layers."

output_path = f"{DIRS['features']}/metrics_per_example_layer_arc.csv"
features_df.to_csv(output_path, index=False)
print(f"\nSaved: {output_path}")
print(features_df.head(10))

# --- Sanity checks ---
print(f"\nCorrect examples: {(features_df.groupby('example_id')['is_correct'].first() == True).sum()}")
print(f"Incorrect examples: {(features_df.groupby('example_id')['is_correct'].first() == False).sum()}")
print(f"\nAny NaNs in metrics?")
print(features_df[["fiedler", "spectral_entropy", "hfer", "smoothness", "dirichlet_energy"]].isna().sum())

print("\n>>> Save & Run All once this looks correct. <<<")

Processing 150 ARC examples...
  [25/150] processed
  [50/150] processed
  [75/150] processed
  [100/150] processed
  [125/150] processed
  [150/150] processed

Done. 150 examples processed successfully, 0 failed.

Feature table shape: (3600, 11)
Expected: 150 examples x 24 layers = 3600 rows

Saved: /kaggle/working/spectral_v2/features/metrics_per_example_layer_arc.csv
     example_id  layer  is_correct  response_len  prompt_len  total_len  \
0  675a81e23cb7      0        True           298          96        394   
1  675a81e23cb7      1        True           298          96        394   
2  675a81e23cb7      2        True           298          96        394   
3  675a81e23cb7      3        True           298          96        394   
4  675a81e23cb7      4        True           298          96        394   
5  675a81e23cb7      5        True           298          96        394   
6  675a81e23cb7      6        True           298          96        394   
7  675a81e23cb7      7     

In [4]:
# ============================================================
# ARC-CHALLENGE — Confound analysis (mirrors GSM8K Checkpoint 9)
# Effect sizes, threshold screening, length confound check.
# No GPU needed.
# ============================================================

In [5]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv(f"{DIRS['features']}/metrics_per_example_layer_arc.csv")
print(f"Loaded ARC feature table: {df.shape}")

METRICS = ["fiedler", "spectral_entropy", "hfer", "smoothness"]

# --- Effect sizes across all 96 layer x metric combinations ---
def cohens_d(a, b):
    n1, n2 = len(a), len(b)
    pooled_std = np.sqrt(((n1-1)*a.std(ddof=1)**2 + (n2-1)*b.std(ddof=1)**2) / (n1+n2-2))
    return 0.0 if pooled_std == 0 else (a.mean() - b.mean()) / pooled_std

effect_rows = []
for metric in METRICS:
    for layer in sorted(df["layer"].unique()):
        layer_df = df[df["layer"] == layer]
        c = layer_df[layer_df["is_correct"] == True][metric]
        i = layer_df[layer_df["is_correct"] == False][metric]
        effect_rows.append({"metric": metric, "layer": layer, "cohens_d": cohens_d(c, i)})

effects_df = pd.DataFrame(effect_rows)
effects_df["abs_d"] = effects_df["cohens_d"].abs()

print("\nTop 10 largest raw effect sizes (ARC):")
print(effects_df.sort_values("abs_d", ascending=False).head(10)[["metric", "layer", "cohens_d"]])

effects_path = f"{DIRS['results']}/arc_raw_effect_sizes.csv"
effects_df.to_csv(effects_path, index=False)
print(f"\nSaved: {effects_path}")

# --- Threshold screening ---
EFFECT_THRESHOLD = 0.5
print(f"\nThreshold screening (|d| >= {EFFECT_THRESHOLD}):")
for metric in METRICS:
    n_qualifying = (effects_df[(effects_df["metric"] == metric)]["abs_d"] >= EFFECT_THRESHOLD).sum()
    print(f"  {metric}: {n_qualifying} layer(s)")

# --- Length confound check on all qualifying layers ---
print(f"\n=== Length confound check ===")
example_level = df.drop_duplicates("example_id")[["example_id", "is_correct", "response_len"]]
print("Response length by correctness:")
print(example_level.groupby("is_correct")["response_len"].describe())

length_t, length_p = stats.ttest_ind(
    example_level[example_level["is_correct"] == True]["response_len"],
    example_level[example_level["is_correct"] == False]["response_len"],
)
print(f"\nLength difference t-test: t={length_t:.3f}, p={length_p:.4f}")

confound_rows = []
for metric in METRICS:
    metric_effects = effects_df[effects_df["metric"] == metric].sort_values("abs_d", ascending=False)
    qualifying = metric_effects[metric_effects["abs_d"] >= EFFECT_THRESHOLD]

    if len(qualifying) == 0:
        continue

    print(f"\n{metric}:")
    for _, row in qualifying.iterrows():
        layer, d_val = int(row["layer"]), row["cohens_d"]
        layer_df = df[df["layer"] == layer]
        r, p = stats.pearsonr(layer_df["response_len"], layer_df[metric])
        flag = abs(r) > 0.3 and p < 0.05
        print(f"  layer {layer} (d={d_val:.2f}): length r={r:.3f}, p={p:.4f}"
              f"{'  <-- FLAGGED' if flag else ''}")
        confound_rows.append({"metric": metric, "layer": layer, "cohens_d": d_val,
                               "length_corr_r": r, "length_corr_p": p, "flagged": flag})

confound_df = pd.DataFrame(confound_rows)
if len(confound_df) > 0:
    confound_path = f"{DIRS['results']}/arc_length_confound_check.csv"
    confound_df.to_csv(confound_path, index=False)
    print(f"\nSaved: {confound_path}")

    survivors = confound_df[~confound_df["flagged"]]
    print(f"\n{len(survivors)} of {len(confound_df)} qualifying candidates survived the length check:")
    print(survivors[["metric", "layer", "cohens_d", "length_corr_r", "length_corr_p"]].to_string(index=False))
else:
    print("\nNo layers cleared the effect-size threshold.")

Loaded ARC feature table: (3600, 11)

Top 10 largest raw effect sizes (ARC):
              metric  layer  cohens_d
14           fiedler     14  0.506898
10           fiedler     10  0.448564
71              hfer     23  0.448074
20           fiedler     20  0.426205
70              hfer     22  0.409281
41  spectral_entropy     17  0.327972
18           fiedler     18  0.317249
50              hfer      2 -0.315930
15           fiedler     15  0.305158
6            fiedler      6  0.284033

Saved: /kaggle/working/spectral_v2/results/arc_raw_effect_sizes.csv

Threshold screening (|d| >= 0.5):
  fiedler: 1 layer(s)
  spectral_entropy: 0 layer(s)
  hfer: 0 layer(s)
  smoothness: 0 layer(s)

=== Length confound check ===
Response length by correctness:
            count        mean        std   min     25%    50%    75%    max
is_correct                                                                 
False        92.0  244.021739  81.047534   2.0  206.00  234.5  261.5  629.0
True         

In [6]:
# ============================================================
# ARC-CHALLENGE — Sink-mass and effective-rank confound check
# On the one length-survivor: fiedler at layer 14
# No GPU needed.
# ============================================================

In [7]:
import os, json
import numpy as np
import pandas as pd
from scipy import stats

ARC_DIR = f"{DIRS['generations']}/arc_challenge"
CANDIDATE_LAYER = 14
MIN_QUERY_POS = 10  # same fix as GSM8K -- avoid causal-mask artifact in sink mass

confound_rows = []
json_files = [f for f in os.listdir(ARC_DIR) if f.endswith(".json")]
print(f"Processing {len(json_files)} examples for sink mass and effective rank...")

for i, fname in enumerate(json_files):
    example_id = fname.replace(".json", "")
    with open(f"{ARC_DIR}/{fname}") as f:
        record = json.load(f)

    tensors = np.load(f"{ARC_DIR}/{example_id}_tensors.npz")
    attentions = tensors["attentions"]
    hidden_states = tensors["hidden_states"]
    seq_len = attentions.shape[2]

    A = attentions[CANDIDATE_LAYER]
    if seq_len > MIN_QUERY_POS:
        sink_mass = A[:, MIN_QUERY_POS:, 0].mean()
    else:
        sink_mass = np.nan

    hs = hidden_states[CANDIDATE_LAYER].astype(np.float64)
    singular_values = np.linalg.svd(hs, compute_uv=False)
    sv_sq = singular_values ** 2
    effective_rank = (sv_sq.sum() ** 2) / (sv_sq ** 2).sum()

    confound_rows.append({
        "example_id": example_id,
        "is_correct": record["is_correct"],
        "sink_mass": float(sink_mass) if not np.isnan(sink_mass) else None,
        "effective_rank": float(effective_rank),
    })

    if (i + 1) % 25 == 0:
        print(f"  [{i+1}/{len(json_files)}] processed")

confound_df = pd.DataFrame(confound_rows).dropna()
print(f"\nDone. n={len(confound_df)}")

extra_path = f"{DIRS['results']}/arc_sink_rank_confounds.csv"
confound_df.to_csv(extra_path, index=False)
print(f"Saved: {extra_path}")

# --- Merge with the fiedler layer-14 values, check correlations ---
features_df = pd.read_csv(f"{DIRS['features']}/metrics_per_example_layer_arc.csv")
layer14 = features_df[features_df["layer"] == CANDIDATE_LAYER][["example_id", "fiedler"]]

merged = layer14.merge(confound_df, on="example_id")
print(f"\n=== fiedler, layer {CANDIDATE_LAYER} (n={len(merged)}) ===")

r_sink, p_sink = stats.pearsonr(merged["sink_mass"], merged["fiedler"])
r_rank, p_rank = stats.pearsonr(merged["effective_rank"], merged["fiedler"])

print(f"vs sink_mass: r={r_sink:.3f}, p={p_sink:.4f}"
      f"{'  <-- POSSIBLE SINK CONFOUND' if abs(r_sink) > 0.3 and p_sink < 0.05 else ''}")
print(f"vs effective_rank: r={r_rank:.3f}, p={p_rank:.4f}"
      f"{'  <-- POSSIBLE RANK CONFOUND' if abs(r_rank) > 0.3 and p_rank < 0.05 else ''}")

sink_flagged = abs(r_sink) > 0.3 and p_sink < 0.05
rank_flagged = abs(r_rank) > 0.3 and p_rank < 0.05

if not sink_flagged and not rank_flagged:
    print("\n>>> SURVIVES all three confound checks (length, sink mass, rank). Real candidate. <<<")
else:
    print("\n>>> Does NOT survive full confound scrutiny. <<<")

Processing 150 examples for sink mass and effective rank...
  [25/150] processed
  [50/150] processed
  [75/150] processed
  [100/150] processed
  [125/150] processed
  [150/150] processed

Done. n=150
Saved: /kaggle/working/spectral_v2/results/arc_sink_rank_confounds.csv

=== fiedler, layer 14 (n=150) ===
vs sink_mass: r=0.106, p=0.1962
vs effective_rank: r=-0.168, p=0.0397

>>> SURVIVES all three confound checks (length, sink mass, rank). Real candidate. <<<


In [8]:
# ============================================================
# ARC-CHALLENGE — Step 9 classifier comparison
# Same 6-feature-set design as GSM8K, leakage-safe nested CV.
# No GPU needed.
# ============================================================

import os
import numpy as np
import pandas as pd

# --- PART 1: Data prep -- build the wide per-example table ---
features_df = pd.read_csv(f"{DIRS['features']}/metrics_per_example_layer_arc.csv")
sink_rank_df = pd.read_csv(f"{DIRS['results']}/arc_sink_rank_confounds.csv")

METRICS = ["fiedler", "spectral_entropy", "hfer", "smoothness"]
N_LAYERS = 24

static_multilayer = features_df.pivot(index="example_id", columns="layer", values=METRICS)
static_multilayer.columns = [f"{m}_L{l}" for m, l in static_multilayer.columns]
static_multilayer = static_multilayer.reset_index()

static_final = features_df[features_df["layer"] == N_LAYERS - 1][["example_id"] + METRICS].copy()
static_final.columns = ["example_id"] + [f"{m}_final" for m in METRICS]

pivot_raw = features_df.pivot(index="example_id", columns="layer", values=METRICS)
delta_rows = {}
for m in METRICS:
    vals = pivot_raw[m].values
    deltas = np.diff(vals, axis=1)
    for l in range(deltas.shape[1]):
        delta_rows[f"{m}_delta_L{l}to{l+1}"] = deltas[:, l]
dynamic_df = pd.DataFrame(delta_rows, index=pivot_raw.index).reset_index()

summary_rows = {}
for m in METRICS:
    vals = pivot_raw[m].values
    deltas = np.diff(vals, axis=1)
    summary_rows[f"{m}_total_variation"] = np.abs(deltas).sum(axis=1)
    summary_rows[f"{m}_max_jump"] = np.abs(deltas).max(axis=1)
    summary_rows[f"{m}_max_jump_layer"] = np.abs(deltas).argmax(axis=1)
dynamic_summary_df = pd.DataFrame(summary_rows, index=pivot_raw.index).reset_index()

labels_df = features_df.drop_duplicates("example_id")[["example_id", "is_correct", "response_len"]].reset_index(drop=True)

# confound features: response_len + sink_mass/effective_rank from layer 14 (the one candidate layer)
confound_wide = sink_rank_df[["example_id", "sink_mass", "effective_rank"]].rename(
    columns={"sink_mass": "sink_mass_L14", "effective_rank": "effective_rank_L14"}
)

master = labels_df.merge(confound_wide, on="example_id", how="inner")
master = master.merge(static_final, on="example_id", how="inner")
master = master.merge(static_multilayer, on="example_id", how="inner")
master = master.merge(dynamic_df, on="example_id", how="inner")
master = master.merge(dynamic_summary_df, on="example_id", how="inner")

print(f"Master table shape: {master.shape}")
assert master["example_id"].nunique() == master.shape[0], "STOP: duplicate example_ids."
print(f"Correct: {(master['is_correct']==True).sum()}, Incorrect: {(master['is_correct']==False).sum()}")

master_path = f"{DIRS['features']}/arc_step9_master_table.csv"
master.to_csv(master_path, index=False)
print(f"Saved: {master_path}")

# --- PART 3: leakage-safe classifier comparison ---
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

CONFOUND_COLS = ["response_len", "sink_mass_L14", "effective_rank_L14"]
STATIC_FINAL_COLS = [c for c in master.columns if c.endswith("_final")]
STATIC_MULTI_COLS = [c for c in master.columns if "_L" in c and "delta" not in c and not c.endswith("_final")
                      and c not in CONFOUND_COLS]
DYNAMIC_RAW_COLS = [c for c in master.columns if "_delta_" in c]
DYNAMIC_SUMMARY_COLS = [c for c in master.columns if ("_total_variation" in c) or ("_max_jump" in c)]
COMBINED_COLS = STATIC_FINAL_COLS + DYNAMIC_SUMMARY_COLS

FEATURE_SETS = {
    "0_confound_only": CONFOUND_COLS,
    "1_static_final": STATIC_FINAL_COLS,
    "2_static_multilayer": STATIC_MULTI_COLS,
    "3_dynamic_raw": DYNAMIC_RAW_COLS,
    "4_dynamic_summary": DYNAMIC_SUMMARY_COLS,
    "5_combined": COMBINED_COLS,
}

print("\nFeature set sizes:")
for name, cols in FEATURE_SETS.items():
    print(f"  {name}: {len(cols)} features")

y = master["is_correct"].astype(int).values
valid_mask = master[CONFOUND_COLS].notna().all(axis=1)
master_clean = master[valid_mask].reset_index(drop=True)
y_clean = master_clean["is_correct"].astype(int).values
print(f"\nSample: n={len(y_clean)}")

N_OUTER_FOLDS = 5
skf = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=42)

def residualize_fold(train_df, test_df, spectral_cols, confound_cols):
    train_resid = train_df[spectral_cols].copy()
    test_resid = test_df[spectral_cols].copy()
    X_train_confound = train_df[confound_cols].values
    X_test_confound = test_df[confound_cols].values
    for col in spectral_cols:
        reg = LinearRegression()
        reg.fit(X_train_confound, train_df[col].values)
        train_resid[col] = train_df[col].values - reg.predict(X_train_confound)
        test_resid[col] = test_df[col].values - reg.predict(X_test_confound)
    return train_resid, test_resid

results = {name: [] for name in FEATURE_SETS}

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(master_clean, y_clean)):
    train_df = master_clean.iloc[train_idx].reset_index(drop=True)
    test_df = master_clean.iloc[test_idx].reset_index(drop=True)
    y_train, y_test = y_clean[train_idx], y_clean[test_idx]

    for name, cols in FEATURE_SETS.items():
        if name == "0_confound_only":
            X_train = train_df[cols].values
            X_test = test_df[cols].values
        else:
            train_resid, test_resid = residualize_fold(train_df, test_df, cols, CONFOUND_COLS)
            X_train = train_resid.values
            X_test = test_resid.values

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        clf = LogisticRegressionCV(Cs=10, cv=3, penalty="l2", max_iter=2000, scoring="roc_auc", random_state=42)
        clf.fit(X_train_scaled, y_train)

        y_pred_proba = clf.predict_proba(X_test_scaled)[:, 1]
        fold_auc = roc_auc_score(y_test, y_pred_proba)
        results[name].append(fold_auc)

    print(f"Fold {fold_idx + 1}/{N_OUTER_FOLDS} done.")

print("\n" + "=" * 60)
print("ARC-CHALLENGE RESULTS: ROC-AUC by feature set")
print("=" * 60)

summary_rows = []
for name in FEATURE_SETS:
    aucs = results[name]
    mean_auc, std_auc = np.mean(aucs), np.std(aucs)
    summary_rows.append({"feature_set": name, "n_features": len(FEATURE_SETS[name]),
                          "mean_auc": mean_auc, "std_auc": std_auc})
    print(f"{name:25s} (n_feat={len(FEATURE_SETS[name]):3d}): AUC = {mean_auc:.3f} +/- {std_auc:.3f}")

summary_df = pd.DataFrame(summary_rows)
summary_path = f"{DIRS['results']}/arc_step9_classifier_comparison.csv"
summary_df.to_csv(summary_path, index=False)
print(f"\nSaved: {summary_path}")

baseline_auc = summary_df[summary_df["feature_set"] == "0_confound_only"]["mean_auc"].values[0]
print(f"\n>>> Confound-only baseline AUC: {baseline_auc:.3f} <<<")
print(">>> Compare each spectral feature set against this baseline. <<<")

Master table shape: (150, 209)
Correct: 58, Incorrect: 92
Saved: /kaggle/working/spectral_v2/features/arc_step9_master_table.csv

Feature set sizes:
  0_confound_only: 3 features
  1_static_final: 4 features
  2_static_multilayer: 96 features
  3_dynamic_raw: 92 features
  4_dynamic_summary: 12 features
  5_combined: 16 features

Sample: n=150
Fold 1/5 done.
Fold 2/5 done.
Fold 3/5 done.
Fold 4/5 done.
Fold 5/5 done.

ARC-CHALLENGE RESULTS: ROC-AUC by feature set
0_confound_only           (n_feat=  3): AUC = 0.568 +/- 0.141
1_static_final            (n_feat=  4): AUC = 0.639 +/- 0.114
2_static_multilayer       (n_feat= 96): AUC = 0.552 +/- 0.054
3_dynamic_raw             (n_feat= 92): AUC = 0.520 +/- 0.101
4_dynamic_summary         (n_feat= 12): AUC = 0.585 +/- 0.106
5_combined                (n_feat= 16): AUC = 0.607 +/- 0.109

Saved: /kaggle/working/spectral_v2/results/arc_step9_classifier_comparison.csv

>>> Confound-only baseline AUC: 0.568 <<<
>>> Compare each spectral feature set

In [9]:
# ============================================================
# ARC-CHALLENGE — Normalized-Laplacian robustness check
# Rerun feature computation using L_sym instead of L, then
# rerun the effect-size + length confound screening, to check
# whether ARC's static-final result holds regardless of which
# Laplacian variant is used.
# No GPU needed -- pure numpy on already-saved tensors.
# ============================================================

import os, json
import numpy as np
import pandas as pd
from scipy import stats

ARC_DIR = f"{DIRS['generations']}/arc_challenge"

def build_normalized_laplacian(A, eps=1e-8):
    heads, seq, _ = A.shape
    W_per_head = 0.5 * (A + A.transpose(0, 2, 1))
    head_mass = W_per_head.sum(axis=(1, 2))
    head_weights = head_mass / (head_mass.sum() + eps)
    W = np.tensordot(head_weights, W_per_head, axes=(0, 0))
    d = W.sum(axis=1)
    d_inv_sqrt = np.where(d > eps, 1.0 / np.sqrt(d + eps), 0.0)
    D_inv_sqrt = np.diag(d_inv_sqrt)
    L_sym = np.eye(seq) - D_inv_sqrt @ W @ D_inv_sqrt
    return L_sym

def compute_spectral_metrics_normalized(L_sym, hidden_state, eps=1e-10):
    n = L_sym.shape[0]
    eigvals, eigvecs = np.linalg.eigh(L_sym)
    eigvals = np.clip(eigvals, 0, None)
    fiedler = eigvals[1] if n > 1 else 0.0
    eigval_sum = eigvals.sum()
    if eigval_sum > eps:
        p = eigvals / eigval_sum
        p_nonzero = p[p > eps]
        spectral_entropy = -np.sum(p_nonzero * np.log(p_nonzero))
        spectral_entropy = spectral_entropy / np.log(n) if n > 1 else 0.0
    else:
        spectral_entropy = 0.0
    x = hidden_state.astype(np.float64)
    signal = np.linalg.norm(x, axis=1)
    signal = (signal - signal.mean()) / (signal.std() + eps)
    dirichlet_energy = float(signal @ L_sym @ signal)
    signal_proj = eigvecs.T @ signal
    energy = signal_proj ** 2
    total_energy = energy.sum()
    cutoff_idx = n // 2
    hfer = float(energy[cutoff_idx:].sum() / (total_energy + eps))
    smoothness = float(1.0 / (1.0 + dirichlet_energy))
    return {"fiedler": float(fiedler), "spectral_entropy": float(spectral_entropy),
            "hfer": hfer, "smoothness": smoothness, "dirichlet_energy": dirichlet_energy}

all_rows = []
json_files = [f for f in os.listdir(ARC_DIR) if f.endswith(".json")]
print(f"Processing {len(json_files)} ARC examples with normalized Laplacian...")

for i, fname in enumerate(json_files):
    example_id = fname.replace(".json", "")
    with open(f"{ARC_DIR}/{fname}") as f:
        record = json.load(f)
    tensors = np.load(f"{ARC_DIR}/{example_id}_tensors.npz")
    attentions = tensors["attentions"]
    hidden_states = tensors["hidden_states"]

    for layer in range(attentions.shape[0]):
        L_sym = build_normalized_laplacian(attentions[layer].astype(np.float64))
        hs = hidden_states[layer]
        metrics = compute_spectral_metrics_normalized(L_sym, hs)
        row = {"example_id": example_id, "layer": layer, "is_correct": record["is_correct"],
               "response_len": record["response_len"], **metrics}
        all_rows.append(row)

    if (i + 1) % 25 == 0:
        print(f"  [{i+1}/{len(json_files)}] processed")

features_norm_df = pd.DataFrame(all_rows)
print(f"\nDone. Shape: {features_norm_df.shape}")
norm_path = f"{DIRS['features']}/metrics_per_example_layer_arc_NORMALIZED.csv"
features_norm_df.to_csv(norm_path, index=False)
print(f"Saved: {norm_path}")

# --- Effect sizes + length confound check on normalized version ---
METRICS = ["fiedler", "spectral_entropy", "hfer", "smoothness"]

def cohens_d(a, b):
    n1, n2 = len(a), len(b)
    pooled_std = np.sqrt(((n1-1)*a.std(ddof=1)**2 + (n2-1)*b.std(ddof=1)**2) / (n1+n2-2))
    return 0.0 if pooled_std == 0 else (a.mean() - b.mean()) / pooled_std

effect_rows = []
for metric in METRICS:
    for layer in sorted(features_norm_df["layer"].unique()):
        layer_df = features_norm_df[features_norm_df["layer"] == layer]
        c = layer_df[layer_df["is_correct"] == True][metric]
        i = layer_df[layer_df["is_correct"] == False][metric]
        effect_rows.append({"metric": metric, "layer": layer, "cohens_d": cohens_d(c, i)})

effects_norm_df = pd.DataFrame(effect_rows)
effects_norm_df["abs_d"] = effects_norm_df["cohens_d"].abs()

print(f"\nTop 10 largest raw effect sizes (NORMALIZED Laplacian):")
print(effects_norm_df.sort_values("abs_d", ascending=False).head(10)[["metric", "layer", "cohens_d"]])

EFFECT_THRESHOLD = 0.5
qualifying = effects_norm_df[effects_norm_df["abs_d"] >= EFFECT_THRESHOLD]
print(f"\n{len(qualifying)} layer x metric combos with |d| >= {EFFECT_THRESHOLD}")
print(qualifying.groupby("metric").size() if len(qualifying) > 0 else "None qualify.")

if len(qualifying) > 0:
    print(f"\nLength confound check on qualifying candidates:")
    for _, row in qualifying.iterrows():
        metric, layer, d_val = row["metric"], int(row["layer"]), row["cohens_d"]
        layer_df = features_norm_df[features_norm_df["layer"] == layer]
        r, p = stats.pearsonr(layer_df["response_len"], layer_df[metric])
        flag = abs(r) > 0.3 and p < 0.05
        print(f"  {metric}, layer {layer} (d={d_val:.2f}): length r={r:.3f}, p={p:.4f}"
              f"{'  <-- FLAGGED' if flag else ''}")

print("\n>>> Compare this survivor set against the combinatorial-Laplacian result (fiedler, layer 14). <<<")

Processing 150 ARC examples with normalized Laplacian...
  [25/150] processed
  [50/150] processed
  [75/150] processed
  [100/150] processed
  [125/150] processed
  [150/150] processed

Done. Shape: (3600, 9)
Saved: /kaggle/working/spectral_v2/features/metrics_per_example_layer_arc_NORMALIZED.csv

Top 10 largest raw effect sizes (NORMALIZED Laplacian):
     metric  layer  cohens_d
14  fiedler     14  0.501183
71     hfer     23  0.418673
10  fiedler     10  0.406870
4   fiedler      4 -0.367157
18  fiedler     18  0.364305
19  fiedler     19  0.297437
15  fiedler     15  0.290916
20  fiedler     20  0.280642
0   fiedler      0 -0.268845
70     hfer     22  0.268715

1 layer x metric combos with |d| >= 0.5
metric
fiedler    1
dtype: int64

Length confound check on qualifying candidates:
  fiedler, layer 14 (d=0.50): length r=-0.299, p=0.0002

>>> Compare this survivor set against the combinatorial-Laplacian result (fiedler, layer 14). <<<
